# Voxel metrics watchboard

Read saved benchmark reports, compare temporal resolutions, and inspect event voxels beside ground truth. **This notebook never launches benchmark scoring or writes reports.** Run all cells once; use **Refresh reports** to watch new results. Raw event files are read only when you click **Show voxel comparison**. No real results are bundled.

Requirements: Python, NumPy, pandas, matplotlib, Pillow, IPython and ipywidgets. If needed, run `%pip install numpy pandas matplotlib pillow ipywidgets` in your notebook environment, then restart the kernel. Open this notebook inside the cloned repository.

## Definitions and provenance

All five reported metrics are errors: **lower is better**. Prediction and GT share the same source crop and time domain $[t_0,t_1)$; neither stream is independently stretched to its observed timestamps. Let $D=t_1-t_0$, $B$ be the full-interval temporal-bin count, and $N$ the number of scored voxels.

| CLI metric | Formula and representation | Attribution |
|---|---|---|
| `pmse_2_v2ce` | $\mathrm{mean}[(P_2(V_p)-P_2(V_g))^2]$; $P_2$ is 3D average pooling with kernel=stride=2 over time, height and width | V2CE, Section III-A |
| `pmse_4_v2ce` | Same formula with kernel=stride=4 | V2CE, Section III-A |
| `voxel_mae` | $N^{-1}\sum_i|S_{p,i}-S_{g,i}|$ | Existing benchmark metric; no paper-specific attribution |
| `voxel_rmse` | $\sqrt{N^{-1}\sum_i(S_{p,i}-S_{g,i})^2}$ | Existing benchmark metric; no paper-specific attribution |
| `voxel_relative_l1` | $\sum_i|S_{p,i}-S_{g,i}|/\sum_i|S_{g,i}|$; undefined for zero GT mass | Existing benchmark metric; no paper-specific attribution |

For legacy metrics, $S$ is the **signed histogram** (ON count minus OFF count) in each uniform bin. MAE/RMSE also export separate ON/OFF errors; these are additional columns, not separate CLI metrics. Signed errors can hide opposite-polarity cancellation.

For PMSE, $V$ has shape $(2,B,H,W)$ with **separate nonnegative ON/OFF channels**. The current implementation uses the forward box-integration interpretation of V2CE II-B: $u=B(t-t_0)/D$, $b=\lfloor u\rfloor$, $f=u-b$; an event contributes $1-f$ to bin $b$ and $f$ to bin $b+1$. Contributions past the final bin are clipped. Pooling never mixes polarities. It uses valid, unpadded blocks and drops incomplete trailing blocks. No per-stream or maximum normalization is applied. Empty-vs-empty PMSE is zero. All retained pooled cells, including inactive cells, have equal weight.

**Protocol adaptation:** the published PMSE formula is used, but the shared full-interval grid, clipping and pooling-edge conventions are benchmark choices. Numerical parity with author code or paper tables is not claimed. PMSE-2/4 retain different temporal/spatial scales. See [the full provenance audit](../VOXEL_METRIC_PROVENANCE.md).

## What bin settings do the papers use?

| Source | Reported setting | Meaning and scope |
|---|---|---|
| [V2CE II-A / III-A](https://arxiv.org/html/2309.08891v2) | 10 temporal bins per frame pair at 30 FPS; 16 pairs per training entry | About 3.33 ms/bin. $16\times10=160$ spans about 0.533 s; **not** 160 bins for every sequence duration. Relevant PMSE source. |
| [V2V 3.2 / 4.1](https://arxiv.org/html/2505.16797v1) | 5 bins per voxel in WebVid training; a discrete voxel spans $B/FPS$ seconds | Model input representation; not a direct GT/generated stream error. |
| [V2E 5.2](https://arxiv.org/html/2006.07722) | 15 slices across a 300 ms N-Caltech101 recording | 20 ms/slice for a recognition experiment; not a PMSE prescription. |
| [EVIS Appendix C, v2](https://arxiv.org/html/2607.08098v2) | 128 bins per polarity across 0.5 s | About 3.91 ms/bin for rotation-speed regression, with per-bin maximum normalization. That normalization is **not** used in our metrics. |
| [EQS Section 4](https://arxiv.org/html/2504.12515v1) | Symbolic $T$ bins with separate polarities | A numeric bin setting and a complete executable RVT evaluator were not verified. EQS is not implemented here. |

Official repositories: [V2CE](https://github.com/ucsd-hdsi-dvs/V2CE-Toolbox), [V2V](https://github.com/HYLZ-2019/V2V), [V2E](https://github.com/SensorsINI/v2e), [EVIS](https://github.com/spikelab-jhu/isaac-sim-event-camera-plugin), [EQS](https://github.com/eventbasedvision/EQS). No voxel constructor is presented as an evaluation metric.

## How to calculate bins

- Frame gap: $1000/FPS$ milliseconds. With $b$ bins per frame gap, bin width is $1000/(FPS\,b)$ ms.
- Whole-interval grid: bin width is $\Delta t=1000D/B$ ms.
- To target width $w$ ms with the current CLI, choose integer $B=\lceil1000D/w\rceil$ and report the **actual** width $1000D/B$. For a 3 s sequence and 30 ms target, $B=100$ exactly.
- The current CLI accepts **bin count**, not milliseconds. One count across unequal-duration sequences gives unequal physical resolutions. A fixed 30 ms setting for each sequence is not currently supported by a single split-wide CLI option. Do not label approximate widths as exact.
- Keep the same bins, bounds and crop for every model/GT comparison. PMSE requires $B,H,W\geq k$; divisibility by $k$ avoids discarded tails.

Example (run in a terminal, outside this watch-only notebook):
```bash
python benchmarks/run_metric.py eds test --metric pmse_2_v2ce --voxel-temporal-bins 160 --methods v2ce v2e
```
Repeat with the other metric names and desired counts. Full-interval metrics use the entire available image array at its existing H×W resolution, without configurable spatial block averaging. Historical frame-pair or spatially averaged reports are excluded below. Native signed V2V files cannot supply timestamp-resolved, polarity-separated PMSE and are not synthesized into event streams. Existing saved predictions may already have a benchmark crop; this notebook cannot recover pixels outside that crop.


In [ ]:
from pathlib import Path
from types import SimpleNamespace
import json
import hashlib
import sys
from html import escape
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'benchmarks/run_metric.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook inside the evtween repository.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from benchmarks import run_metric as runner

# EDIT THESE SETTINGS, then rerun all cells.
REPORT_ROOT = ROOT / 'reports'
PRED_ROOT = ROOT / 'data_test_evt'
DATA_ROOT = None  # Path containing dataset/split/sequence; None uses saved input.json source.
SPLIT = 'test'
DATASETS = None   # None = discover all; otherwise e.g. ['eds', 'bsergb'].
BIN_COUNTS = None # None = discover all; otherwise e.g. [40, 80, 160, 320].
METHODS = None    # None = matched methods; otherwise list explicit method folder names.
TOP_FOUR_ONLY = False  # True keeps the union of the top four rank levels in each column.
# Optional exact run directory, keyed by (dataset, metric, bins, spatial_block).
RUN_OVERRIDE = {}
METRICS = ['pmse_2_v2ce', 'pmse_4_v2ce', 'voxel_mae', 'voxel_rmse', 'voxel_relative_l1']
PALETTE = ['#164b80', '#75a9d6', '#b4d4ed', '#e0eef8']


## Benchmark comparisons

Click **Refresh reports** to read saved results. These are the only tables displayed. Each table is one dataset and temporal-bin count; the four best distinct rank levels use the blue palette, darkest first. Ties share a rank. Means weight shared sequences equally; `n` is the matched sequence count. Methods and sequences are matched across selected bin counts for each metric/channel. Incomplete, historical frame-pair and spatially averaged results are excluded. Incompatible protocol/code versions require an explicit `RUN_OVERRIDE`.

**Signed is not the average of ON and OFF errors.** Write $e_+=P_+-G_+$ and $e_-=P_--G_-$. Signed MAE is $\mathrm{mean}|e_+-e_-|$; signed RMSE is $\sqrt{\mathrm{mean}(e_+-e_-)^2}$. The per-polarity columns evaluate $e_+$ and $e_-$ separately. Their ordering need not agree: correlated errors may cancel in Signed even when both polarity errors are large. For squared errors, $\mathrm{MSE}_{signed}=\mathrm{MSE}_{ON}+\mathrm{MSE}_{OFF}-2\mathrm{mean}(e_+e_-)$. Lower Signed error alone does not establish more accurate separate polarities.

Example: GT has (ON, OFF)=(10,10). Prediction A=(12,8) has polarity errors 2 and 2, but Signed error 4. Prediction B=(15,15) has polarity errors 5 and 5, but Signed error 0. Both MAE and RMSE have these values for this single voxel. This demonstrates the mechanism; the screenshot alone cannot establish error correlation in the actual runs.

The input is the full available H×W image at every temporal bin. PMSE's pooling is part of its published formula. This notebook does not alter metric definitions or stored scores.


In [ ]:
def load_reports():
    candidates, inventory = {}, []
    for cfg_path in sorted(REPORT_ROOT.glob('*/*/*/config.json')):
        try:
            cfg = json.loads(cfg_path.read_text(encoding='utf-8'))
            metric, dataset = cfg.get('metric'), cfg.get('dataset')
            if cfg.get('split') != SPLIT or metric not in METRICS:
                continue
            if DATASETS is not None and dataset not in DATASETS:
                continue
            options = cfg.get('options', {})
            bins = options.get('temporal_bins')
            csv_path = cfg_path.with_name('results.csv')
            if not csv_path.is_file():
                continue
            raw = pd.read_csv(csv_path)
            legacy = options.get('legacy_frame_voxels', False) or bins is None or options.get('block_size', 1) != 1
            inventory.append(dict(dataset=dataset, metric=metric, bins=bins,
                excluded_historical=legacy,
                statuses=json.dumps(raw.status.value_counts().to_dict()), run=str(cfg_path.parent)))
            if legacy or (BIN_COUNTS is not None and bins not in BIN_COUNTS):
                continue
            key = (dataset, metric, int(bins), int(options.get('block_size', 1)))
            candidates.setdefault(key, []).append((csv_path, cfg, raw))
        except (OSError, ValueError, KeyError, AttributeError) as exc:
            print(f'Skipped unreadable/in-progress report {cfg_path}: {exc}')
    selected = {}
    for key, choices in candidates.items():
        override = RUN_OVERRIDE.get(key)
        if override is not None:
            choices = [c for c in choices if c[0].parent.name == override]
            if not choices:
                raise ValueError(f'Unknown run override {override!r}: {key}')
        fingerprints = {json.dumps({k: v for k, v in cfg.items()
                        if k not in ('dataset', 'split')}, sort_keys=True) for _, cfg, _ in choices}
        if len(fingerprints) != 1:
            raise ValueError(f'Ambiguous protocol/code versions for {key}; set RUN_OVERRIDE. '
                             + str([p.parent.name for p, _, _ in choices]))
        path, cfg, rows = max(choices, key=lambda c: (c[0].stat().st_mtime_ns, str(c[0])))
        if rows.duplicated(['sequence', 'method']).any():
            raise ValueError(f'Duplicate sequence/method rows: {path}')
        selected[key] = dict(path=path, cfg=cfg, rows=rows)
    missing = set(RUN_OVERRIDE) - set(selected)
    if missing:
        raise ValueError(f'Overrides did not select a report: {missing}')
    if BIN_COUNTS is not None:
        for dataset in sorted({key[0] for key in selected}):
            for metric in METRICS:
                absent = set(BIN_COUNTS) - {b for d, m, b, _ in selected if d == dataset and m == metric}
                if absent:
                    print(f'Missing requested settings: {dataset}/{metric}, bins={sorted(absent)}')
    return selected, pd.DataFrame(inventory)

def matched_summary(reports):
    records, coverage = [], []
    families = sorted({(d, metric, block) for d, metric, bins, block in reports})
    for dataset, metric, block in families:
        group = {bins: item for (d, m, bins, b), item in reports.items()
                 if (d, m, b) == (dataset, metric, block)}
        # Comparisons across bins must use the same implementation/crop/protocol.
        comparable = set()
        for item in group.values():
            cfg = dict(item['cfg']); opt = dict(cfg.get('options', {})); opt.pop('temporal_bins', None)
            if opt.get('block_size', 1) == 1:
                opt.pop('block_size', None)  # Older full-image reports stored an explicit identity option.
            cfg['options'] = opt
            comparable.add(json.dumps(cfg, sort_keys=True))
        if len(comparable) > 1:
            raise ValueError(f'Incompatible configurations across bins for {(dataset, metric, block)}')
        fields = ['value', 'value_on', 'value_off'] if metric in ('voxel_mae', 'voxel_rmse') else ['value']
        for field in fields:
            frames = {}
            for bins, item in group.items():
                rows = item['rows'].copy()
                rows[field] = pd.to_numeric(rows.get(field, np.nan), errors='coerce')
                rows = rows[rows.status.eq('ok') & np.isfinite(rows[field]) & rows.method.ne('ground_truth')]
                if METHODS is not None:
                    rows = rows[rows.method.isin(METHODS)]
                frames[bins] = rows
            sets = [set(rows.method) for rows in frames.values()]
            methods = sorted(set.intersection(*sets)) if sets else []
            observed = set.union(*sets) if sets else set()
            excluded = (set(METHODS) if METHODS is not None else observed) - set(methods)
            seq_sets = [set(rows.loc[rows.method.eq(method), 'sequence'])
                        for rows in frames.values() for method in methods]
            shared = set.intersection(*seq_sets) if seq_sets else set()
            coverage.append(dict(dataset=dataset, metric=metric, channel=field,
                bins=sorted(group), methods=methods, excluded_methods=sorted(excluded),
                matched_sequences=len(shared)))
            for bins, rows in frames.items():
                means = rows[rows.sequence.isin(shared) & rows.method.isin(methods)].groupby('method')[field].mean()
                ranks = means.rank(method='dense', ascending=True)
                for method in means.index:
                    records.append(dict(dataset=dataset, metric=metric, bins=bins, spatial_block=block,
                        channel=field, method=method, mean=float(means[method]), rank=int(ranks[method]),
                        n=len(shared), sequences=tuple(sorted(shared))))
    return pd.DataFrame(records), pd.DataFrame(coverage)

def show_tables(summary):
    if summary.empty:
        print('No complete matched cohorts yet. Check report paths, selected bins and completed score statuses.'); return
    for (dataset, bins), rows in summary.groupby(['dataset', 'bins'], sort=True):
        cols = list(dict.fromkeys((r.metric, r.channel, r.spatial_block, r.n) for r in rows.itertuples()))
        methods = sorted(rows.method.unique())
        if TOP_FOUR_ONLY:
            methods = sorted(rows.loc[rows['rank'] <= 4, 'method'].unique())
        html = [f'<h3>{escape(dataset)} · {bins} temporal bins · {escape(SPLIT)}</h3>',
                '<div style="overflow-x:auto"><table><thead><tr><th>Method</th>']
        for metric, channel, block, n in cols:
            channel_label = {'value': 'ON + OFF' if metric.startswith('pmse_') else 'Signed',
                             'value_on': 'ON', 'value_off': 'OFF'}[channel]
            html.append(f'<th>{escape(metric)} ↓<br>{channel_label}<br>n={n}</th>')
        html.append('</tr></thead><tbody>')
        for method in methods:
            html.append(f'<tr><th>{escape(method)}</th>')
            for metric, channel, block, n in cols:
                match = rows[(rows.method == method) & (rows.metric == metric) &
                             (rows.channel == channel) & (rows.spatial_block == block)]
                if match.empty:
                    html.append('<td>—</td>'); continue
                item = match.iloc[0]; rank = int(item['rank']); style = ''
                if rank <= 4:
                    style = f'background:{PALETTE[rank-1]};color:' + ('white' if rank == 1 else '#102b43')
                html.append(f'<td style="{style}" title="Rank {rank}; equal sequence weight">{item["mean"]:.6g}</td>')
            html.append('</tr>')
        display(HTML(''.join(html) + '</tbody></table></div>'))

REPORTS, SUMMARY = {}, pd.DataFrame()
report_output = widgets.Output()
refresh_button = widgets.Button(description='Refresh reports', icon='refresh')
def refresh_reports(_=None):
    global REPORTS, SUMMARY
    with report_output:
        clear_output(wait=True)
        try:
            REPORTS, inventory = load_reports()
            SUMMARY, coverage = matched_summary(REPORTS)
            if not coverage.empty and (coverage.matched_sequences == 0).any():
                print('Some comparisons have no shared complete sequences and are omitted.')
            show_tables(SUMMARY)
        except Exception as exc:
            REPORTS, SUMMARY = {}, pd.DataFrame()
            print(f'Reports were not combined: {exc}')
display(refresh_button, report_output)
refresh_button.on_click(refresh_reports)
refresh_reports()


## Voxel comparison: GT and models

White background; **ON = red**, **OFF = blue**, **both = purple**. Both polarities appear in the same panel. Panels use three columns per row, with GT first. Each panel represents one method for the selected sequence. No additional error tables, error-image rows or trend charts are displayed.

2D shows one temporal bin. Optional 3D shows voxel locations across the selected number of consecutive bins, with time in milliseconds on the vertical axis. The full source interval determines the grid; selecting a view does not realign either stream. PMSE mode uses the forward-weighted voxel input before pooling; histogram mode uses polarity-separated counts. Both are diagnostics, not new metrics.

All panels share the same intensity/size scale and camera. Purple means both polarity channels occupy the voxel; they are not subtracted. 2D intensity encodes total magnitude, and 3D marker size encodes it. For rendering only, each 3D panel is deterministically limited to `MAX_3D_POINTS` occupied voxel locations; the caption reports shown/total locations. This display cap does not change scores. Native V2V is omitted because separate polarities and timestamps cannot be recovered from its signed frame arrays. Source/prediction metadata are validated. Existing saved crops are preserved.


In [ ]:
def temporal_slice(events, start, stop, bins, index, height, width, representation):
    # Memory bounded to two H×W planes; same full-domain weights as voxel_metric.
    if not 0 <= index < bins:
        raise ValueError('Bin index must lie between 0 and bins - 1.')
    t = np.asarray(events['t'])
    keep = (t >= start) & (t < stop)
    u = (t[keep]-start)/(stop-start)*bins
    u = np.minimum(u, np.nextafter(float(bins), -np.inf))
    b = np.floor(u).astype(np.int64); f = u-b
    ids = (np.asarray(events['p'])[keep] <= 0).astype(np.int64)*height*width
    ids += np.asarray(events['y'])[keep].astype(np.int64)*width + np.asarray(events['x'])[keep].astype(np.int64)
    weights = (b == index).astype(float)
    if representation == 'pmse':
        weights = (b == index)*(1-f) + (b+1 == index)*f
    return np.bincount(ids, weights=weights, minlength=2*height*width).reshape(2, height, width)

MAX_3D_POINTS = 25000

def show_voxel(dataset, sequence, bins, index, representation, method_names, use_3d=False, depth=8):
    from matplotlib.patches import Patch
    if bins < 1 or not 0 <= index < bins or depth < 1:
        raise ValueError('Use positive bins/depth and a bin index in [0, bins).')
    split_dir = (Path(DATA_ROOT) if DATA_ROOT is not None else
                 runner.v2.old.setting('EVTWEEN_DATA', runner.v2.old.LAB / 'evtween/data')) / dataset / SPLIT
    prediction_dir = PRED_ROOT/dataset/SPLIT
    folders = sorted(p for p in prediction_dir.glob('*') if p.is_dir())
    names = sorted({p.name for d in folders for p in d.iterdir() if p.is_dir()})
    if not sequence:
        sequence = next((name for name in names if any((d/name/'events.npz').is_file() for d in folders)), None)
    if not sequence:
        raise ValueError('No saved event streams found; set paths/dataset/sequence.')
    args = SimpleNamespace(dataset=dataset, split=SPLIT, data_root=DATA_ROOT)
    source = runner.source_for(sequence, split_dir, folders, args)
    seq = runner.Sequence(source)
    edges, meta = seq.geometry
    start, stop = float(edges[0]), float(edges[-1])
    duration = stop-start
    print(f'{dataset}/{sequence} | {meta["width"]} × {meta["height"]} pixels | '
          f'{bins} bins over {duration:.6g} s | actual width {duration*1000/bins:.6g} ms')
    streams = {'GT': seq.gt}
    requested = [s.strip() for s in method_names.split(',') if s.strip()] if method_names.strip() else METHODS
    for folder in folders:
        if requested is not None and folder.name not in requested:
            continue
        if folder.name == 'v2v':
            print('Omitted native V2V: separate polarity/time information unavailable.'); continue
        file = folder/sequence/'events.npz'
        if not file.is_file():
            print(f'Missing events: {folder.name}'); continue
        try:
            runner.verify_metadata(folder/sequence/'input.json', meta, args, sequence)
            streams[folder.name] = runner.read_events(file, meta)
        except Exception as exc:
            print(f'Omitted {folder.name}: {exc}')
    if requested is not None:
        for method in sorted(set(requested)-set(streams)-{'v2v'}):
            print(f'Unavailable requested method: {method}')
    h, w = meta['height'], meta['width']
    indices = range(index, min(bins, index+depth)) if use_3d else [index]
    data, scale = {}, 0.0
    for name, events in streams.items():
        if use_3d:
            coordinates, on_values, off_values = [], [], []
            for b in indices:
                planes = temporal_slice(events,start,stop,bins,b,h,w,representation)
                y,x = np.nonzero(planes.sum(axis=0)>0)
                coordinates.append(np.column_stack([x,y,np.full(len(x), b)]))
                on_values.append(planes[0,y,x]); off_values.append(planes[1,y,x])
            xyz=np.concatenate(coordinates); on=np.concatenate(on_values); off=np.concatenate(off_values)
            if len(on):
                scale=max(scale,float((on+off).max()))
            data[name]=(xyz,on,off)
        else:
            planes=temporal_slice(events,start,stop,bins,index,h,w,representation)
            scale=max(scale,float(planes.sum(axis=0).max()))
            data[name]=planes
    scale=scale or 1.0
    nrows=int(np.ceil(len(data)/3))
    fig=plt.figure(figsize=(15,4.8*nrows),facecolor='white',layout='constrained')
    colors=np.array([[0.90,0.12,0.12],[0.10,0.30,0.90],[0.65,0.12,0.75]])
    for i,(name,values) in enumerate(data.items()):
        ax=fig.add_subplot(nrows,3,i+1,projection='3d' if use_3d else None)
        ax.set_facecolor('white')
        if use_3d:
            xyz,on,off=values; total=len(xyz)
            keep=np.linspace(0,total-1,min(total,MAX_3D_POINTS),dtype=int) if total else np.array([],dtype=int)
            xyz,on,off=xyz[keep],on[keep],off[keep]
            category=np.where((on>0)&(off>0),2,np.where(on>0,0,1))
            ax.scatter(xyz[:,0],xyz[:,1],(xyz[:,2]+.5)*duration*1000/bins+start*1000,
                       c=colors[category],s=3+24*np.sqrt((on+off)/scale),alpha=.75,depthshade=False)
            ax.set(xlim=(0,w),ylim=(h,0),zlim=((start+index*duration/bins)*1000,
                       (start+min(bins,index+depth)*duration/bins)*1000),zlabel='Time (ms)')
            ax.view_init(elev=23,azim=-65)
            for axis in (ax.xaxis,ax.yaxis,ax.zaxis):
                axis.set_pane_color((1,1,1,1))
            ax.set_title(f'{name} · {len(keep):,}/{total:,} occupied voxels',color='black')
        else:
            on,off=values; mass=on+off
            category=np.where((on>0)&(off>0),2,np.where(on>0,0,1))
            alpha=np.sqrt(mass/scale)[...,None]
            rgb=1-alpha*(1-colors[category])
            ax.imshow(rgb,origin='upper',vmin=0,vmax=1)
            ax.set_title(name,color='black')
        ax.set_xlabel('x (pixels)',color='black'); ax.set_ylabel('y (pixels)',color='black')
        ax.tick_params(colors='black')
    fig.legend(handles=[Patch(color=c,label=l) for c,l in zip(colors,['ON','OFF','ON + OFF overlap'])],
               loc='outside upper center',ncol=3,facecolor='white',labelcolor='black')
    plt.show(); plt.close(fig)

detail_dataset = widgets.Text(value=(DATASETS[0] if DATASETS else 'eds'), description='Dataset:')
detail_sequence = widgets.Text(value='', description='Sequence:')
detail_bins = widgets.BoundedIntText(value=160,min=1,max=1000000,description='Bins:')
detail_index = widgets.IntText(value=0,description='First bin:')
detail_repr = widgets.Dropdown(options=[('PMSE forward weights','pmse'),('Histogram','histogram')],description='Input:')
detail_3d = widgets.Checkbox(value=False,description='3D view')
detail_depth = widgets.BoundedIntText(value=8,min=1,max=1000,description='3D bins:')
detail_methods = widgets.Text(value='',description='Methods:',placeholder='Comma-separated; blank = all')
detail_button = widgets.Button(description='Show voxel comparison',icon='image')
detail_output = widgets.Output()
def render_detail(_=None):
    with detail_output:
        clear_output(wait=True)
        try:
            show_voxel(detail_dataset.value.strip(),detail_sequence.value.strip(),detail_bins.value,
                       detail_index.value,detail_repr.value,detail_methods.value,detail_3d.value,detail_depth.value)
        except Exception as exc:
            print(f'Voxel view unavailable: {exc}')
detail_button.on_click(render_detail)
display(widgets.VBox([widgets.HBox([detail_dataset,detail_sequence]),
    widgets.HBox([detail_bins,detail_index]),widgets.HBox([detail_repr,detail_3d,detail_depth]),
    detail_methods,detail_button]),detail_output)


## Interpretation checklist

- Inspect matched `n` before reading the rankings; missing or unfinished runs are not zeros.
- Use identical bin counts and physical domains for model comparisons. Across datasets, also examine actual milliseconds per bin.
- PMSE smooths both space and time. Compare PMSE-2 and PMSE-4 as different scales, not interchangeable numbers.
- The voxel slice uses the entire sequence's time grid; changing the displayed bin index does not re-align either stream.
- Saved reports are read-only. Re-run **Refresh reports** after changing filters or after scoring completes. To add another temporal resolution, run the benchmark CLI separately.
